# Soulie: treinamento da validação de fotos

Pipeline: **foto → recorte 224x224 → MobileNetV2 (embedding) → classificador + detector de anomalia → regra de decisão**.

Este notebook:
1. Carrega as fotos e gera os embeddings
2. Avalia o classificador com validação cruzada
3. Calibra o limite de distância (anomalia) e o limite de confiança
4. Treina os modelos finais e salva tudo em `modelos/`

Pastas esperadas em `dataset/`: `bicicleta`, `transporte_publico`, `carro_particular` e `invalidas` (esta última só para calibração, nunca entra no treino).

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors

import preprocessamento as pp

PASTA_DATASET = Path("dataset")
PASTA_MODELOS = Path("modelos")
PASTA_MODELOS.mkdir(exist_ok=True)

CLASSES = ["bicicleta", "transporte_publico", "carro_particular"]
CLASSES_VALIDAS = ["bicicleta", "transporte_publico"]
CLASSE_NEGATIVA = "carro_particular"

K_VIZINHOS = 3                 # quantas fotos válidas mais parecidas entram na distância
N_FOLDS = 5                    # validação cruzada
TAXA_REJEICAO_VALIDAS = 0.05   # % de fotos válidas que aceitamos barrar como "cena inválida"
SEED = 42
VERSAO = "1.0"

## 1. Carregar as fotos

In [ ]:
fotos, rotulos = [], []
for classe in CLASSES:
    caminhos = pp.listar_fotos(PASTA_DATASET / classe)
    print(f"{classe:20s} {len(caminhos)} fotos")
    for caminho in caminhos:
        fotos.append(pp.abrir_imagem(caminho))
        rotulos.append(classe)
rotulos = np.array(rotulos)

invalidas = [pp.abrir_imagem(c) for c in pp.listar_fotos(PASTA_DATASET / "invalidas")]
print(f"{'invalidas':20s} {len(invalidas)} fotos (só calibração)")

assert len(fotos) > 0, "Nenhuma foto encontrada em dataset/"
if len(invalidas) == 0:
    print("Aviso: sem fotos em dataset/invalidas. A calibração vai usar só as fotos de carro.")

## 2. Gerar os embeddings

Cada foto vira um vetor de 1280 números. Também geramos as variações (espelhada, mais clara, mais escura, zoom), que entram **só no treino**.

In [ ]:
emb = pp.gerar_embeddings(fotos)

variacoes = [pp.aumentar(f) for f in fotos]
N_VAR = len(variacoes[0])
emb_aug = pp.gerar_embeddings([v for vs in variacoes for v in vs]).reshape(len(fotos), N_VAR, -1)

emb_invalidas = pp.gerar_embeddings(invalidas)
print("Embeddings:", emb.shape, "| variações:", emb_aug.shape, "| inválidas:", emb_invalidas.shape)

In [ ]:
def montar_treino(indices):
    """Junta as fotos originais com as variações delas."""
    X = np.concatenate([emb[indices], emb_aug[indices].reshape(-1, emb.shape[1])])
    y = np.concatenate([rotulos[indices], np.repeat(rotulos[indices], N_VAR)])
    return X, y


def novo_classificador():
    return LogisticRegression(max_iter=2000, class_weight="balanced")


def novo_detector(X_validas):
    return NearestNeighbors(n_neighbors=K_VIZINHOS).fit(X_validas)

## 3. Validação cruzada

Cada foto é avaliada por um modelo que **não a viu** no treino. Isso dá números honestos mesmo com poucas fotos.

In [ ]:
ORDEM = sorted(CLASSES)  # ordem das colunas do predict_proba
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

probs_oof = np.zeros((len(fotos), len(ORDEM)))
dist_oof = np.zeros(len(fotos))

for treino, teste in skf.split(emb, rotulos):
    X, y = montar_treino(treino)
    clf = novo_classificador().fit(X, y)
    assert list(clf.classes_) == ORDEM
    probs_oof[teste] = clf.predict_proba(emb[teste])

    det = novo_detector(X[np.isin(y, CLASSES_VALIDAS)])
    dist_oof[teste] = pp.distancia_anomalia(det, emb[teste])

pred_oof = np.array(ORDEM)[probs_oof.argmax(axis=1)]
print(classification_report(rotulos, pred_oof, digits=3))

ConfusionMatrixDisplay(confusion_matrix(rotulos, pred_oof, labels=ORDEM), display_labels=ORDEM).plot(
    cmap="Greens", xticks_rotation=20
)
plt.title("Matriz de confusão (validação cruzada)")
plt.show()

## 4. Modelos finais (treinados com todas as fotos)

In [ ]:
X_todas, y_todas = montar_treino(np.arange(len(fotos)))
clf_final = novo_classificador().fit(X_todas, y_todas)
det_final = novo_detector(X_todas[np.isin(y_todas, CLASSES_VALIDAS)])

# As inválidas nunca entram no treino, então podem ser avaliadas pelo modelo final
probs_inv = clf_final.predict_proba(emb_invalidas) if len(invalidas) else np.empty((0, len(ORDEM)))
dist_inv = pp.distancia_anomalia(det_final, emb_invalidas) if len(invalidas) else np.empty(0)

## 5. Limite de distância (anomalia)

Escolhido para barrar no máximo `TAXA_REJEICAO_VALIDAS` das fotos válidas. Depois vemos quanto das fotos inválidas e de carro fica acima dele.

In [ ]:
eh_valida = np.isin(rotulos, CLASSES_VALIDAS)
eh_carro = rotulos == CLASSE_NEGATIVA

limite_distancia = float(np.quantile(dist_oof[eh_valida], 1 - TAXA_REJEICAO_VALIDAS))
print(f"Limite de distância: {limite_distancia:.4f}")
print(f"Válidas barradas pela distância:   {(dist_oof[eh_valida] > limite_distancia).mean():.0%}")
print(f"Carros barrados pela distância:    {(dist_oof[eh_carro] > limite_distancia).mean():.0%}")
if len(invalidas):
    print(f"Inválidas barradas pela distância: {(dist_inv > limite_distancia).mean():.0%}")

plt.figure(figsize=(8, 4))
plt.hist(dist_oof[eh_valida], bins=20, alpha=0.6, label="válidas")
plt.hist(dist_oof[eh_carro], bins=20, alpha=0.6, label="carro")
if len(invalidas):
    plt.hist(dist_inv, bins=20, alpha=0.6, label="inválidas")
plt.axvline(limite_distancia, color="black", linestyle="--", label="limite")
plt.xlabel("distância até as fotos válidas")
plt.ylabel("quantidade de fotos")
plt.legend()
plt.show()

## 6. Limite de confiança

Simula a regra de decisão completa para vários limites e escolhe o que melhor equilibra **aprovar fotos válidas** e **barrar fraudes** (carro + inválidas).

In [ ]:
def simular(limite_confianca):
    cfg = {
        "limite_distancia": limite_distancia,
        "limite_confianca": limite_confianca,
        "classe_negativa": CLASSE_NEGATIVA,
    }
    status = np.array([pp.decidir(p, ORDEM, d, cfg)["status"] for p, d in zip(probs_oof, dist_oof)])
    status_inv = np.array([pp.decidir(p, ORDEM, d, cfg)["status"] for p, d in zip(probs_inv, dist_inv)])

    aprovacao_validas = (status[eh_valida] == "APROVADO").mean()
    fraudes = np.concatenate([status[eh_carro], status_inv])
    bloqueio_fraudes = (fraudes == "REPROVADO").mean()
    return aprovacao_validas, bloqueio_fraudes


print(f"{'limite':>7} | {'válidas aprovadas':>17} | {'fraudes barradas':>16} | {'equilíbrio':>10}")
resultados = []
for limite in np.round(np.arange(0.40, 0.96, 0.05), 2):
    aprov, bloq = simular(limite)
    equilibrio = (aprov + bloq) / 2
    resultados.append((equilibrio, limite, aprov, bloq))
    print(f"{limite:7.2f} | {aprov:17.0%} | {bloq:16.0%} | {equilibrio:10.3f}")

# Em caso de empate, fica com o limite mais alto (mais rigoroso)
_, limite_confianca, aprov, bloq = max(resultados)
limite_confianca = float(limite_confianca)
print(f"\nLimite de confiança escolhido: {limite_confianca:.2f}")
print(f"Válidas aprovadas: {aprov:.0%} | Fraudes barradas: {bloq:.0%}")

## 7. Salvar os modelos

In [ ]:
config = {
    "versao": VERSAO,
    "tamanho_imagem": pp.TAMANHO_IMAGEM,
    "limite_confianca": round(limite_confianca, 4),
    "limite_distancia": round(limite_distancia, 4),
    "k_vizinhos": K_VIZINHOS,
    "classes_modelo": ORDEM,
    "classes_validas": CLASSES_VALIDAS,
    "classe_negativa": CLASSE_NEGATIVA,
    "metricas_validacao": {
        "validas_aprovadas": round(float(aprov), 4),
        "fraudes_barradas": round(float(bloq), 4),
        "fotos_treino": int(len(fotos)),
        "fotos_invalidas_calibracao": int(len(invalidas)),
    },
}

joblib.dump(clf_final, PASTA_MODELOS / "classificador.joblib")
joblib.dump(det_final, PASTA_MODELOS / "detector_anomalia.joblib")
(PASTA_MODELOS / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(config, indent=2, ensure_ascii=False))

## 8. Teste rápido (mesmo caminho que a API faz)

In [ ]:
caminho_teste = pp.listar_fotos(PASTA_DATASET / "bicicleta")[0]  # troque por qualquer foto

img = pp.abrir_imagem(caminho_teste)
e = pp.gerar_embeddings([img])
d = float(pp.distancia_anomalia(det_final, e)[0])
p = clf_final.predict_proba(e)[0]
print(caminho_teste.name, "→", pp.decidir(p, list(clf_final.classes_), d, config))